In [ ]:

import os
os.environ["JAVA_HOME"] = os.path.expanduser("~/.sdkman/candidates/java/current")
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
os.environ["SPARK_LOCAL_DIRS"] = "/mnt/bigdata/spark_temp"

os.makedirs("/mnt/bigdata/spark_temp", exist_ok=True)
os.makedirs("/mnt/bigdata/data", exist_ok=True)
os.makedirs("/mnt/bigdata/models", exist_ok=True)
os.makedirs("/mnt/bigdata/output", exist_ok=True)

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CIC-IDS-Intrusion-Detection") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.local.dir", "/mnt/bigdata/spark_temp") \
    .config("spark.sql.warehouse.dir", "/mnt/bigdata/spark_warehouse") \
    .getOrCreate()

print("Spark Version:", spark.version)
print(" SparkSession created successfully!")

Spark Version: 4.1.1
 SparkSession created successfully!


26/02/25 09:58:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [ ]:

from datasets import load_dataset
import pandas as pd

print(" Loading dataset from Hugging Face...")


dataset = load_dataset(
    "auliraff/CIC-IDS-Collection",
    split="train",
    cache_dir="/mnt/bigdata/data/hf_cache"  # Cache to mounted disk
)

print(f" Dataset loaded!")
print(f" Total rows: {len(dataset)}")
print(f" Total columns: {len(dataset.column_names)}")
print(f" Columns: {dataset.column_names}")

print("\n Saving as Parquet to /mnt/bigdata/data...")
dataset.to_parquet("/mnt/bigdata/data/cic_ids_raw.parquet")
print(" Saved to /mnt/bigdata/data/cic_ids_raw.parquet")


import os
size = os.path.getsize("/mnt/bigdata/data/cic_ids_raw.parquet")
print(f" File size: {size / (1024**3):.2f} GB")

 Loading dataset from Hugging Face...
 Dataset loaded!
 Total rows: 9167581
 Total columns: 59
 Columns: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'SYN Flag Count', 'URG Flag Count', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd B

Creating parquet from Arrow format:   0%|          | 0/31 [00:00<?, ?ba/s]

 Saved to /mnt/bigdata/data/cic_ids_raw.parquet
 File size: 0.99 GB


In [ ]:
df = spark.read.parquet("/mnt/bigdata/data/cic_ids_raw.parquet")
print(f" Rows: {df.count()}")
print(f" Columns: {len(df.columns)}")
df.printSchema()
df.show(5, truncate=True)

 Rows: 9167581
 Columns: 59
root
 |-- Flow Duration: long (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Backward Packets: integer (nullable = true)
 |-- Fwd Packets Length Total: double (nullable = true)
 |-- Bwd Packets Length Total: double (nullable = true)
 |-- Fwd Packet Length Max: double (nullable = true)
 |-- Fwd Packet Length Mean: float (nullable = true)
 |-- Fwd Packet Length Std: float (nullable = true)
 |-- Bwd Packet Length Max: double (nullable = true)
 |-- Bwd Packet Length Mean: float (nullable = true)
 |-- Bwd Packet Length Std: float (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |-- Flow Packets/s: double (nullable = true)
 |-- Flow IAT Mean: float (nullable = true)
 |-- Flow IAT Std: float (nullable = true)
 |-- Flow IAT Max: double (nullable = true)
 |-- Flow IAT Min: double (nullable = true)
 |-- Fwd IAT Total: double (nullable = true)
 |-- Fwd IAT Mean: float (nullable = true)
 |-- Fwd IAT Std: float (nullable = tr

26/02/25 05:52:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+-----------------+----------------------+------------------------+------------------------+---------------------+----------------------+---------------------+---------------------+----------------------+---------------------+----------------+----------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-----------------+-----------------+-------------+-------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+---------------+--------------------+--------------------+-------------------+-----------------+-------------------+-----------------+------------------+------------------+--------------------+----------------+-----------+----------+----------+----------+---------+--------+--------+--------+------+----------+
|Flow Duration|Total Fwd Packets|Total Backward Pa

In [ ]:
print("\n SCHEMA VALIDATION")
print(f"   Total Rows    : {df.count():,}")
print(f"   Total Columns : {len(df.columns)}")
print(f"   Storage Format: Parquet (columnar, compressed,")
print(f"                   best for Spark SQL queries)")


 SCHEMA VALIDATION
   Total Rows    : 9,167,581
   Total Columns : 59
   Storage Format: Parquet (columnar, compressed,
                   best for Spark SQL queries)


In [ ]:
from pyspark.sql import functions as F
print("\nMISSING VALUE VALIDATION")
missing_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

missing_cols = {k: v for k, v in missing_counts.items() if v > 0}
if missing_cols:
    print(f"   Columns with nulls: {missing_cols}")
else:
    print(" No missing values found ")


MISSING VALUE VALIDATION


[Stage 11:==================================================>       (7 + 1) / 8]

 No missing values found 


In [ ]:
from pyspark.sql import functions as F
print("\n 3. CLASS DISTRIBUTION (Target Variable)")
print("\n   ClassLabel (8 broad classes):")
df.groupBy("ClassLabel").count() \
  .orderBy("count", ascending=False) \
  .show(truncate=False)
print("   Label (33 fine-grained attack types):")
df.groupBy("Label").count() \
  .orderBy("count", ascending=False) \
  .show(50, truncate=False)


 3. CLASS DISTRIBUTION (Target Variable)

   ClassLabel (8 broad classes):


+------------+-------+
|ClassLabel  |count  |
+------------+-------+
|Benign      |7186189|
|DDoS        |1234729|
|DoS         |397344 |
|Botnet      |145968 |
|Bruteforce  |103244 |
|Infiltration|94857  |
|Webattack   |2995   |
|Portscan    |2255   |
+------------+-------+

   Label (33 fine-grained attack types):
+--------------------+-------+
|Label               |count  |
+--------------------+-------+
|Benign              |7186189|
|DDoS-LOIC-HTTP      |575364 |
|DoS-Hulk            |318740 |
|DDoS-HOIC           |198861 |
|Botnet              |145968 |
|DDoS                |128062 |
|DDoS-NTP            |121328 |
|DDoS-TFTP           |98833  |
|Bruteforce-SSH      |97260  |
|Infiltration        |94857  |
|DoS-Goldeneye       |52324  |
|DDoS-Syn            |47757  |
|DDoS-UDP            |28863  |
|DoS-Slowloris       |15243  |
|DDoS-MSSQL          |11784  |
|DDoS-UDPLag         |8452   |
|Bruteforce-FTP      |5984   |
|DoS-Slowhttptest    |5271   |
|DDoS-Ddossim        |5115   |


In [ ]:

print("\n PARTITIONING STRATEGY")
print("   Partitioning by ClassLabel for query optimization")
df_partitioned = df.repartition(8, "ClassLabel")
df_partitioned.write \
    .mode("overwrite") \
    .partitionBy("ClassLabel") \
    .parquet("/mnt/bigdata/data/cic_ids_partitioned")

print("    Saved partitioned parquet to /mnt/bigdata/data/cic_ids_partitioned")
print("   Justification: Partitioning by ClassLabel speeds up")
print("   model training queries that filter by attack type")



 PARTITIONING STRATEGY
   Partitioning by ClassLabel for query optimization


[Stage 22:==================================================>       (7 + 1) / 8]

    Saved partitioned parquet to /mnt/bigdata/data/cic_ids_partitioned
   Justification: Partitioning by ClassLabel speeds up
   model training queries that filter by attack type


In [ ]:

print("\n DUPLICATE VALIDATION")
total = df.count()
distinct = df.distinct().count()
duplicates = total - distinct
print(f"   Total rows    : {total:,}")
print(f"   Distinct rows : {distinct:,}")
print(f"   Duplicates    : {duplicates:,}")

print("\n 6. INITIAL STATISTICS (Numeric Features)")
df.select([
    "Flow Duration", "Total Fwd Packets",
    "Total Backward Packets", "Flow Bytes/s",
    "Flow Packets/s"
]).describe().show(truncate=False)


 DUPLICATE VALIDATION


   Total rows    : 9,167,581
   Distinct rows : 9,167,271
   Duplicates    : 310

 6. INITIAL STATISTICS (Numeric Features)


[Stage 32:==================================================>       (7 + 1) / 8]

+-------+--------------------+-----------------+----------------------+-------------------+------------------+
|summary|Flow Duration       |Total Fwd Packets|Total Backward Packets|Flow Bytes/s       |Flow Packets/s    |
+-------+--------------------+-----------------+----------------------+-------------------+------------------+
|count  |9167581             |9167581          |9167581               |9167581            |9167581           |
|mean   |1.5906685544299526E7|40.79504986102659|9.505221388281162     |2854875.5219017146 |11029.195097372141|
|stddev |6.569715058155395E8 |2066.283169026518|580.5652471058053     |6.354385178751393E7|104054.69377068832|
|min    |-919011000000       |0                |0                     |-2.61E8            |-2000000.0        |
|max    |120000000           |309629           |291922                |2.944E9            |4000000.0         |
+-------+--------------------+-----------------+----------------------+-------------------+------------------+



In [ ]:

print(" ADVANCED DATA CLEANING & PREPROCESSING")



print("\n INITIAL DATASET OVERVIEW")
print(f"Total Rows: {df.count():,}")
print(f"Total Columns: {len(df.columns)}")
print("\nSchema:")
df.printSchema()


 ADVANCED DATA CLEANING & PREPROCESSING

 INITIAL DATASET OVERVIEW
Total Rows: 9,167,581
Total Columns: 59

Schema:
root
 |-- Flow Duration: long (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Backward Packets: integer (nullable = true)
 |-- Fwd Packets Length Total: double (nullable = true)
 |-- Bwd Packets Length Total: double (nullable = true)
 |-- Fwd Packet Length Max: double (nullable = true)
 |-- Fwd Packet Length Mean: float (nullable = true)
 |-- Fwd Packet Length Std: float (nullable = true)
 |-- Bwd Packet Length Max: double (nullable = true)
 |-- Bwd Packet Length Mean: float (nullable = true)
 |-- Bwd Packet Length Std: float (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |-- Flow Packets/s: double (nullable = true)
 |-- Flow IAT Mean: float (nullable = true)
 |-- Flow IAT Std: float (nullable = true)
 |-- Flow IAT Max: double (nullable = true)
 |-- Flow IAT Min: double (nullable = true)
 |-- Fwd IAT Total: double (nullable =

In [ ]:
print("\n REMOVING DATA LEAKAGE COLUMNS")

leakage_cols = ["Flow ID", "Source IP", "Destination IP", "Timestamp"]

existing_leakage = [c for c in leakage_cols if c in df.columns]
print("Columns identified for removal:", existing_leakage)

df_clean = df.drop(*existing_leakage)

print(f"Remaining columns after removal: {len(df_clean.columns)}")


 REMOVING DATA LEAKAGE COLUMNS
Columns identified for removal: []
Remaining columns after removal: 59


In [ ]:


print("\n REPLACING INFINITE VALUES")


numeric_cols = [c for c, t in df_clean.dtypes if t in ['double', 'float', 'int', 'bigint']]
print(f"Numeric Columns Detected: {len(numeric_cols)}")

df_clean = df_clean.select([
    F.when(F.col(c).isin([float("inf"), float("-inf")]), None)
     .otherwise(F.col(c))
     .alias(c)
    if c in numeric_cols else F.col(c)
    for c in df_clean.columns
])

print("Infinite values replaced with NULL.")



 REPLACING INFINITE VALUES
Numeric Columns Detected: 54
Infinite values replaced with NULL.


In [ ]:


print("\n NULL VALUE CHECK (BEFORE IMPUTATION)")

null_before = df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).collect()[0].asDict()

null_cols_before = {k: v for k, v in null_before.items() if v > 0}

if null_cols_before:
    print("Columns with missing values:")
    for k, v in null_cols_before.items():
        print(f"{k}: {v:,}")
else:
    print("No missing values detected.")



 NULL VALUE CHECK (BEFORE IMPUTATION)


[Stage 8:===================================================>       (7 + 1) / 8]

No missing values detected.


In [ ]:


print("\n APPLYING MEDIAN IMPUTATION")

median_dict = {}

for col_name in numeric_cols:
    median_value = df_clean.approxQuantile(col_name, [0.5], 0.01)[0]
    median_dict[col_name] = median_value
    print(f"Median for {col_name}: {median_value}")

df_clean = df_clean.fillna(median_dict)

print("Median imputation completed.")


 APPLYING MEDIAN IMPUTATION


Median for Flow Duration: 389882.0


Median for Total Fwd Packets: 3.0


Median for Total Backward Packets: 2.0


Median for Fwd Packets Length Total: 97.0


...

Median for Subflow Fwd Packets: 3.0


Median for Subflow Fwd Bytes: 97.0
Median for Subflow Bwd Packets: 2.0


Median for Subflow Bwd Bytes: 231.0
Median for Init Fwd Win Bytes: 2049.0


Median for Init Bwd Win Bytes: 123.0
Median for Fwd Act Data Packets: 1.0
Median for Fwd Seg Size Min: 20.0
Median for Active Mean: 0.0
Median for Active Std: 0.0
Median for Active Max: 0.0
Median for Active Min: 0.0
Median for Idle Mean: 0.0
Median for Idle Std: 0.0


Median for Idle Max: 0.0
Median for Idle Min: 0.0
Median imputation completed.


In [ ]:
print("\n NULL VALUE CHECK (AFTER IMPUTATION)")

null_after = df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).collect()[0].asDict()

null_cols_after = {k: v for k, v in null_after.items() if v > 0}

if null_cols_after:
    print("WARNING: Some columns still contain nulls:", null_cols_after)
else:
    print("No remaining null values.")


 NULL VALUE CHECK (AFTER IMPUTATION)


[Stage 173:>                                                        (0 + 5) / 8]

No remaining null values.


In [ ]:
print("\n DUPLICATE CHECK")

before = df_clean.count()
df_clean = df_clean.dropDuplicates()
after = df_clean.count()

print(f"Total rows before removing duplicates: {before:,}")
print(f"Total rows after removing duplicates : {after:,}")
print(f"Duplicates removed: {before - after:,}")



 DUPLICATE CHECK


[Stage 181:==================================================>    (23 + 2) / 25]

Total rows before removing duplicates: 9,167,581
Total rows after removing duplicates : 9,167,271
Duplicates removed: 310


In [ ]:
print("\n CREATING BINARY TARGET")

df_clean = df_clean.withColumn(
    "target_binary",
    F.when(F.col("ClassLabel") == "BENIGN", 0).otherwise(1)
)

print("Binary target created: BENIGN = 0, ATTACK = 1")

print("\n Binary Class Distribution:")
df_clean.groupBy("target_binary").count().orderBy("count", ascending=False).show()



 CREATING BINARY TARGET
Binary target created: BENIGN = 0, ATTACK = 1

 Binary Class Distribution:


[Stage 187:============================================>          (20 + 5) / 25]

+-------------+-------+
|target_binary|  count|
+-------------+-------+
|            1|9167271|
+-------------+-------+



In [ ]:
from pyspark.ml.feature import StringIndexer
print("\n ENCODING MULTI-CLASS TARGET")
label_indexer = StringIndexer(
    inputCol="ClassLabel",
    outputCol="target_multiclass"
)

df_clean = label_indexer.fit(df_clean).transform(df_clean)

print("Multi-class target created.")
print("\n Multi-class Distribution:")
df_clean.groupBy("ClassLabel").count().orderBy("count", ascending=False).show(20)


 ENCODING MULTI-CLASS TARGET


Multi-class target created.

 Multi-class Distribution:


[Stage 203:============================================>          (20 + 5) / 25]

+------------+-------+
|  ClassLabel|  count|
+------------+-------+
|      Benign|7185881|
|        DDoS|1234727|
|         DoS| 397344|
|      Botnet| 145968|
|  Bruteforce| 103244|
|Infiltration|  94857|
|   Webattack|   2995|
|    Portscan|   2255|
+------------+-------+



In [ ]:
print("\n FEATURE SKEWNESS ANALYSIS")

skewness_df = df_clean.select([
    F.skewness(c).alias(c)
    for c in numeric_cols
])

skewness_df.show()

print("Note: |Skewness| > 1 indicates high skew (consider log transform later).")


 FEATURE SKEWNESS ANALYSIS


[Stage 209:====================================================>  (24 + 1) / 25]

+-------------------+-----------------+----------------------+------------------------+------------------------+---------------------+----------------------+---------------------+---------------------+----------------------+---------------------+-----------------+-----------------+-------------------+-----------------+-----------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+-----------------+------------------+-----------------+-----------------+-----------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+----------------------+------------------+--------------------+--------------------+-------------------+-----------------+-------------------+-----------------+------------------+------------------+--------------------+-------------------+-----------------+-----------------+-----------------+------------------+--------------

In [ ]:
print("\n DESCRIPTIVE STATISTICS (KEY NUMERIC FEATURES)")


key_features = numeric_cols[:5]
df_clean.select(key_features).describe().show()


 DESCRIPTIVE STATISTICS (KEY NUMERIC FEATURES)


+-------+-------------------+-----------------+----------------------+------------------------+------------------------+
|summary|      Flow Duration|Total Fwd Packets|Total Backward Packets|Fwd Packets Length Total|Bwd Packets Length Total|
+-------+-------------------+-----------------+----------------------+------------------------+------------------------+
|  count|            9167271|          9167271|               9167271|                 9167271|                 9167271|
|   mean|1.590722344247661E7| 40.7963686248612|     9.505533107944556|      2063.8951149147874|      10011.181785833538|
| stddev|6.569826072695055E8|2066.318092953906|     580.5750607263669|       83587.21193906463|      1281318.9101978543|
|    min|      -919011000000|                0|                     0|                     0.0|                     0.0|
|    max|          120000000|           309629|                291922|            1.44391846E8|             6.5545303E8|
+-------+-------------------+---

In [ ]:
print("\n FINAL CLEAN DATASET SUMMARY")
print(f"Final Row Count   : {df_clean.count():,}")
print(f"Final Column Count: {len(df_clean.columns)}")

print("\nSample Data:")
df_clean.show(5, truncate=False)


 FINAL CLEAN DATASET SUMMARY


Final Row Count   : 9,167,271
Final Column Count: 61

Sample Data:


[Stage 227:>                                                        (0 + 1) / 1]

+-------------+-----------------+----------------------+------------------------+------------------------+---------------------+----------------------+---------------------+---------------------+----------------------+---------------------+----------------+-----------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-----------------+-----------------+-------------+-------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+---------------+--------------------+--------------------+-------------------+-----------------+-------------------+-----------------+------------------+------------------+--------------------+----------------+-----------+----------+-----------+----------+-----------+-----------+-----------+-----------+------+----------+-------------+-----------------+
|Flow

In [ ]:
print("\n CACHING CLEANED DATASET")

df_clean = df_clean.persist()

print("Dataset persisted in memory.")

print(" PREPROCESSING COMPLETE")



 CACHING CLEANED DATASET
Dataset persisted in memory.
 PREPROCESSING COMPLETE


In [ ]:
print("\n Saving cleaned dataset to disk...")

df_clean.write \
    .mode("overwrite") \
    .parquet("/mnt/bigdata/data/cic_ids_cleaned")

print(" Cleaned dataset saved successfully.")


 Saving cleaned dataset to disk...


26/02/25 06:13:56 WARN MemoryStore: Not enough space to cache rdd_564_43 in memory! (computed 43.9 MiB so far)
26/02/25 06:13:56 WARN BlockManager: Persisting block rdd_564_43 to disk instead.
26/02/25 06:13:57 WARN MemoryStore: Not enough space to cache rdd_564_43 in memory! (computed 43.9 MiB so far)
26/02/25 06:13:57 WARN MemoryStore: Not enough space to cache rdd_564_44 in memory! (computed 43.9 MiB so far)
26/02/25 06:13:57 WARN BlockManager: Persisting block rdd_564_44 to disk instead.
26/02/25 06:13:57 WARN MemoryStore: Not enough space to cache rdd_564_44 in memory! (computed 43.9 MiB so far)
26/02/25 06:13:59 WARN MemoryStore: Not enough space to cache rdd_564_2 in memory! (computed 43.9 MiB so far)
26/02/25 06:14:27 WARN MemoryStore: Not enough space to cache rdd_564_43 in memory! (computed 43.9 MiB so far)
26/02/25 06:14:27 WARN MemoryStore: Not enough space to cache rdd_564_44 in memory! (computed 43.9 MiB so far)
[Stage 232:=================================================

 Cleaned dataset saved successfully.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
df_clean = spark.read.parquet("/mnt/bigdata/data/cic_ids_cleaned")
df_clean = df_clean.withColumn(
    "target_binary",
    F.when(F.lower(F.col("ClassLabel")) == "benign", 0).otherwise(1)
)

print(f"df_clean loaded: {df_clean.count():,} rows, {len(df_clean.columns)} columns")
print("Ready to run remaining cells.")

df_clean loaded: 9,167,271 rows, 61 columns
Ready to run remaining cells.


In [ ]:
from pyspark.sql.functions import broadcast

label_map = spark.createDataFrame([
    ("Benign", 0), ("DDoS", 1), ("DoS", 2),
    ("Botnet", 3), ("Bruteforce", 4),
    ("Infiltration", 5), ("Webattack", 6), ("Portscan", 7)
], ["ClassLabel", "ClassID"])

df_with_id = df_clean.join(
    broadcast(label_map),
    on="ClassLabel",
    how="left"
)

print("Broadcast join applied.")
print("Justification: label_map is tiny (<1KB), broadcasting avoids")
print("shuffle of the 9M row DataFrame across executors.")
print(f"Rows after join: {df_with_id.count():,}")

Broadcast join applied.
Justification: label_map is tiny (<1KB), broadcasting avoids
shuffle of the 9M row DataFrame across executors.


[Stage 6:==================================>                       (6 + 4) / 10]

Rows after join: 9,167,271


In [ ]:
import datetime

print("DATA LINEAGE LOG")


try:
    df_validate = spark.read.parquet("/mnt/bigdata/data/cic_ids_cleaned")
    row_count = df_validate.count()
    col_count = len(df_validate.columns)

    assert row_count > 9000000, f"Row count too low: {row_count}"
    assert col_count >= 59,     f"Column count too low: {col_count}"
    assert "ClassLabel"    in df_validate.columns, "Missing ClassLabel"
    assert "target_binary" in df_validate.columns, "Missing target_binary"

    print(f"  Source      : HuggingFace auliraff/CIC-IDS-Collection")
    print(f"  Raw rows    : 9,167,581")
    print(f"  After dedup : {row_count:,} (310 duplicates removed)")
    print(f"  Columns     : {col_count}")
    print(f"  Format      : Parquet (columnar, compressed)")
    print(f"  Partitioned : by ClassLabel (8 partitions)")
    print(f"  Validated at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Status      :  PASSED all integrity checks")

except AssertionError as e:
    print(f" PIPELINE VALIDATION FAILED: {e}")
    raise
except Exception as e:
    print(f" UNEXPECTED ERROR: {e}")
    raise

DATA LINEAGE LOG
  Source      : HuggingFace auliraff/CIC-IDS-Collection
  Raw rows    : 9,167,581
  After dedup : 9,167,271 (310 duplicates removed)
  Columns     : 61
  Format      : Parquet (columnar, compressed)
  Partitioned : by ClassLabel (8 partitions)
  Validated at: 2026-02-25 10:03:23
  Status      :  PASSED all integrity checks


In [ ]:
print(f"Shuffle partitions : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Driver memory      : {spark.conf.get('spark.driver.memory')}")
print(f"Data partitions    : {df_clean.rdd.getNumPartitions()}")

Shuffle partitions : 50
Driver memory      : 4g
Data partitions    : 10
